# Models: trivial baseline

This notebook fits the first baseline and scores it on the validation split. The logistic regression and the latent factor model reuse the same interface and the same two metrics, so every row of the final results table is comparable.

The baseline gives every row the class frequencies of the training set, so it always answers "fit". It sets the floor: a model that cannot beat it has learned nothing.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))

import pandas as pd  # noqa: E402

from src import data, metrics, models  # noqa: E402

## Data and split

`src.data.load()` reads the raw file, or the small sample when `DATA_DIR` points at `tests/fixtures`. `src.data.split()` returns the shared 80/10/10 split with seed 158. This notebook leaves the test split alone; the final evaluation scores it once.

In [ ]:
df = data.load()
train, valid, test = data.split(df)
print(f"train {len(train)}, valid {len(valid)}")
train["fit"].value_counts(normalize=True).round(3)

## Fit and score

Every model returns a probability for each class, because the primary metric (average one-vs-rest AUC) ranks scores. Macro-F1 uses the hard labels, which are the most likely class. The majority baseline gives the same probabilities to every row, so its AUC is 0.5 by construction. Macro-F1 shows what always saying "fit" costs: the two minority classes score 0.

In [ ]:
baseline = models.MajorityClass().fit(train)
results = pd.DataFrame(
    {
        "average AUC": [metrics.average_auc(valid["fit"], baseline.predict_proba(valid))],
        "macro-F1": [metrics.macro_f1(valid["fit"], baseline.predict(valid))],
    },
    index=["MajorityClass"],
).round(4)
print(results)
results

## Adding a model

A new model subclasses `models.Model`, implements `fit(train)` and `predict_proba(df)`, and gets `predict` for free. Swap `MajorityClass()` for it in the scoring cell and nothing else changes.